# Beginner 03: Secure Research Agent

**Level:** Beginner · **Duration:** 60–90 min · **Prerequisites:** Security Foundations (Beginner 01), Prompt Injection (Beginner 02)

## 1. Scenario and Objectives
This module builds directly upon Beginner 01 and Beginner 02. In this lesson, we build a credential-free enterprise research assistant. We will combine tool policy, provenance, and data sensitivity to demonstrate that **retrieved content = evidence**, not instructions or permission to act.

Our employee needs to ask the assistant internal questions (like retention policies). The corpus contains various document types (public, internal, confidential, poisoned). We will build a pipeline that safely retrieves, filters, generates, and validates answers.

In [ ]:
import sys, importlib
from pathlib import Path
for p in [Path("."), Path("curriculum/beginner/03-secure-research-agent")]:
    if (p / "03_secure_research_agent.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break
lab = importlib.import_module("03_secure_research_agent")
print("Loaded module successfully.")

## 2. Threat Model
When building a research agent, several attacks are possible:
- **Prompt Injection**: A retrieved document contains instructions designed to hijack the model.
- **Data Exfiltration**: The model leaks confidential documents to unauthorized users.
- **Unauthorized Actions**: The model uses tools outside its allowed scope.
- **Citation Laundering**: The model fabricates citations to look authoritative while making unsupported claims.

## 3. Metadata-Rich Corpus
Let's look at the documents available. Notice that they have explicit `Provenance` and `Sensitivity` metadata.

In [ ]:
for doc_id, doc in lab.DOCUMENT_STORE.items():
    print(f"{doc_id}: {doc.sensitivity.name} | {doc.provenance.name} | {doc.title}")

## 4. Why Heuristic Filtering is Insufficient
We have a detection heuristic to flag obvious injections. It helps observability, but attackers easily bypass these filters. Therefore, it **must never be the authorization boundary**.

In [ ]:
print("Obvious poison detected?", lab.detect_suspicious_content(lab.DOCUMENT_STORE["doc-poison-obvious"].text))
print("Bypass poison detected?", lab.detect_suspicious_content(lab.DOCUMENT_STORE["doc-poison-bypass"].text))

## 5. Least Privilege Capability Contract
The agent must operate under strict least privilege. The capability contract defines exactly what it is allowed to do.

In [ ]:
print("Allowed operations:", lab.ALLOWED_OPERATIONS)

## 6. Access / Sensitivity Context
Retrieval itself is an authorization boundary. We don't retrieve confidential data and hope the model hides it; we filter it *before* the model ever sees it. Data minimization!

In [ ]:
alice_ctx = lab.ResearchContext("alice", frozenset({lab.Sensitivity.PUBLIC, lab.Sensitivity.INTERNAL}))
bob_ctx = lab.ResearchContext("bob", frozenset({lab.Sensitivity.PUBLIC, lab.Sensitivity.INTERNAL, lab.Sensitivity.CONFIDENTIAL}))
print(f"Alice can see: {[s.name for s in alice_ctx.allowed_sensitivities]}")
print(f"Bob can see:   {[s.name for s in bob_ctx.allowed_sensitivities]}")

## 7. Retrieval Authorization Check
Let's see what happens when Alice queries for the production reporting secret.

In [ ]:
auth_docs, blocked_docs = lab.RetrievalService.get_authorized_evidence("production reporting secret", alice_ctx)
print(f"Authorized: {[d.document_id for d in auth_docs]}")
print(f"Blocked: {[d.document_id for d in blocked_docs]}")

## 8. Untrusted Model Output
Because prompt injection is possible, the model output (`ModelOutput`) is completely untrusted until validated by the deterministic application logic.

## 9. Deterministic Validators
The `PolicyEngine` enforces three deterministic checks:
1. **Capability**: Is the proposed action allowed?
2. **Citations**: Does every cited document exist and was it authorized for this user?
3. **Grounding**: Does the cited document actually support the generated claim?

## 10. Normal Execution
Let's run a normal, safe query for Alice.

In [ ]:
agent = lab.SecureResearchAgent(alice_ctx)
ans, audit = agent.answer_query("ticket retention policy")
print(f"Terminal state: {audit.terminal_state}\nAnswer: {ans}")

## 11. Poisoned Evidence Cannot Authorize Actions
If Alice's search retrieves the obvious poisoned document, the model might try to `reveal_secret`. But the Capability Contract blocks it.

In [ ]:
ans, audit = agent.answer_query("Read user profile")
print(f"Terminal state: {audit.terminal_state} ({audit.reason})\nAnswer: {ans}")

## 12. Poisoned Trusted Source
Even if a document's provenance is `INTERNAL`, it only serves as `INFORMATIONAL` evidence. It cannot grant operational authority. Here, a legacy document tries to authorize an email.

In [ ]:
ans, audit = agent.answer_query("Check legacy operations")
print(f"Terminal state: {audit.terminal_state} ({audit.reason})\nAnswer: {ans}")

## 13. Handling Insufficient Evidence
If no authorized evidence supports the query, we fail closed. The model is not allowed to hallucinate an answer without evidence.

In [ ]:
ans, audit = agent.answer_query("What color is the sky?")
print(f"Terminal state: {audit.terminal_state} ({audit.reason})\nAnswer: {ans}")

## 14. Secret Data Minimization
When Alice asks for the API key, the retrieval boundary blocks the confidential document. It never reaches the model, ensuring the secret is safe. The audit log also safely excludes the contents.

In [ ]:
ans, audit = agent.answer_query("What is the production reporting secret?")
print(f"Terminal state: {audit.terminal_state} ({audit.reason})\nAnswer: {ans}")
print(f"Secret leaked? {'DEMO_API_KEY_12345' in str(audit.to_dict())}")

## 15. Validation: Citation Laundering
Citation laundering occurs when a model produces an incorrect/malicious claim, but cites a valid, trusted document to look authoritative.

In [ ]:
ans, audit = agent.answer_query("launder retention policy")
print(f"Terminal state: {audit.terminal_state} ({audit.reason})\nAnswer: {ans}")

## 16. Adversarial Matrix
Run the complete demo to see how the system handles all edge cases safely.

In [ ]:
lab.run_demo()

## 17. Exercises
1. **New Sensitivity Tier:** Add a `RESTRICTED` sensitivity level and test access controls.
2. **New Prohibited Capability:** Add an `upload_file` tool and prove poisoned evidence cannot authorize it.
3. **Citation Validation:** Create an answer that cites a non-retrieved document. Ensure the system rejects it.
4. **Poisoned Trusted Source:** Prove that malicious text inside a trusted document cannot grant operational authority.
5. **Audit Redaction:** Ensure a synthetic confidential secret never appears in the audit output.

## 18. Production Upgrades
This simulation teaches boundaries. In production:
- The corpus is a Vector Database.
- Retrieval uses Semantic Search.
- The Capability Contract is enforced by Policy-as-Code (e.g. OPA, Cedar).
- Grounding validation uses Entailment models (NLI) or specific claim-checking LLMs.